# Spark Exercises 04 — Window Functions

Window functions are the single most useful Spark skill for real Foundry work, and they are awkward or limited in pandas/Polars. A window lets you compute a value for each row **relative to a group of related rows** — without collapsing the rows like `groupBy` does. Think: *ranking*, *running totals*, *“previous row”*, *“each row’s share of its group”*.

**Data file:** `data/orders.csv`

### 1. **Setup** — open a local `SparkSession` (already written for you). In Foundry the session is created for you; locally we open one ourselves. `F` is the functions module — almost every column expression lives there.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("spark-course")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### 2. Read `data/orders.csv`, keep only positive `quantity`, and add `revenue = quantity * unit_price`. Call it `o`.

### 3. Import the `Window` class.

### 4. **Number each customer's orders chronologically.** Define a window partitioned by `customer_id`, ordered by `order_ts`, and add `order_seq = row_number()`. Show `customer_id`, `order_ts`, `order_seq` for 12 rows.

### 5. **Top orders per channel.** Define a window partitioned by `channel`, ordered by `revenue` descending, add `rnk = rank()`, and keep only the **top 3** orders in each channel.

### 6. **`rank` vs `dense_rank` — the difference shows up only on ties.** `revenue` is almost unique, so instead rank each **customer's** orders by `quantity` (an integer — lots of ties). Add both `rank()` and `dense_rank()` and show one customer: after a tie, `rank` **skips** the next number(s) while `dense_rank` keeps counting without gaps.

### 7. **Running total.** For each customer, compute the cumulative sum of `revenue` over time. Use the `w_time` window with `rowsBetween(Window.unboundedPreceding, Window.currentRow)`.

### 8. **Compare to the previous order.** Add `prev_revenue = lag(revenue)` over `w_time`, and `delta = revenue - prev_revenue`. Show 12 rows.

### 9. **Each order's share of its customer's total.** Define a window partitioned by `customer_id` **without** `orderBy` (so it spans the whole partition), and compute `pct = revenue / sum(revenue) over that window`, as a percentage.

### 10. **Moving average.** For each customer, compute a 3-order moving average of `revenue` (current row + 2 previous) with `rowsBetween(-2, 0)`.

### 11. **One row per customer: their biggest order.** Use `row_number()` over `w_rev_cust` (partition by `customer_id`, order by `revenue` desc) and keep only `rn == 1`. Show 10 customers and their top order.

### 12. **Split each customer into spend quartiles.** Add `quartile = ntile(4)` over `w_rev_cust`. Show 12 rows.